# Config-Driven Training Tutorial

This tutorial shows how to use configuration dictionaries (hyperparameter files) to drive the full training workflow in `kgcnn_torch`. It covers:

- **HyperParameter**: A class for loading and managing training configs from `.json` or `.py` files
- **Config structure**: Model, data, and training sections
- **Full training workflow**: Load config, create model, create optimizer, run `fit()`
- **Scheduler configuration**: via `get_scheduler()`

This mirrors the Keras KGCNN config-based training approach but uses native PyTorch components.

## 1. Config File Structure

The hyperparameter dictionary has the following top-level sections:

```python
hyper = {
    "info": {
        # General information for the training run
        "postfix": "",
        "kgcnn_version": "1.0.0"
    },
    "model": {
        "config": {
            # Model-specific parameters (architecture, dimensions, etc.)
        }
    },
    "data": {
        "dataset": {
            "class_name": "QM9Dataset",
            "config": {}
        }
    },
    "training": {
        "fit": {
            # Training loop parameters (epochs, batch_size, etc.)
        },
        "compile": {
            # Optimizer and loss configuration
        },
        "scheduler": {
            # Learning rate scheduler configuration
        },
        "scaler": {
            # Target scaling configuration
        },
        "cross_validation": {
            # K-fold cross-validation parameters
        }
    }
}
```


In [ ]:
import json
import torch
import torch.nn as nn
import numpy as np

## 2. Building a Hyperparameter Config Step by Step

In [ ]:
hyper = {}

### 2.1 Model Section

The model config specifies the architecture. In `kgcnn_torch`, models are native `nn.Module` classes (e.g., `GCNModel`, `SchNetModel`). The config keys correspond to constructor arguments.

In [ ]:
hyper["model"] = {
    "class_name": "GCNModel",
    "module_name": "kgcnn_torch.models.gcn",
    "config": {
        "node_dim": 64,
        "depth": 3,
        "gcn_units": 100,
        "gcn_activation": "relu",
        "node_pooling": "sum",
        "output_units": [64, 32],
        "output_activation": "relu",
        "output_final_activation": "linear",
        "num_targets": 1,
        "output_embedding": "graph",
        "use_node_embedding": True,
        "num_embeddings": 95,
    }
}
print("Model config:", json.dumps(hyper["model"], indent=2))

### 2.2 Data Section

The data section identifies the dataset class and any preprocessing methods to apply.

In [ ]:
hyper["data"] = {
    "data_unit": "eV",
    "dataset": {
        "class_name": "QM9Dataset",
        "module_name": "kgcnn_torch.data.datasets.QM9Dataset",
        "config": {},
        "target_index": 10
    }
}


### 2.3 Training Section

This section configures the optimizer, loss, scheduler, cross-validation, and fit parameters. Unlike Keras KGCNN which serializes Keras objects, `kgcnn_torch` uses plain dictionaries with class names and parameters.

In [ ]:
hyper["training"] = {
    "cross_validation": {
        "class_name": "KFold",
        "config": {"n_splits": 5, "random_state": 42, "shuffle": True}
    },
    "scaler": {
        "class_name": "StandardScaler",
        "config": {"with_std": True, "with_mean": True}
    },
    "compile": {
        "optimizer": {
            "class_name": "Adam",
            "config": {"lr": 0.001}
        },
        "loss": "mean_squared_error"
    },
    "scheduler": {
        "name": "warmup_exponential",
        "config": {
            "warmup_epochs": 10,
            "decay_rate": 0.96,
            "decay_epochs": 10
        }
    },
    "fit": {
        "batch_size": 32,
        "epochs": 200,
        "verbose": 1
    }
}

### 2.4 Info Section

In [ ]:
hyper["info"] = {
    "postfix": "_v1",
    "kgcnn_version": "1.0.0"
}

In [ ]:
# View the complete config
print(json.dumps(hyper, indent=2))

## 3. HyperParameter Class

The `HyperParameter` class wraps a config dictionary and provides convenient property accessors for each section. It can load from a Python dict, a `.json` file, or a `.py` file.

In [ ]:
from kgcnn_torch.training.hyper import HyperParameter

# Create from dict
hp = HyperParameter(hyper, model_name="GCN", dataset_name="QM9")
print(hp)


In [ ]:
# Access individual sections
print("Model config:", hp.model_config)
print("\nFit config:", hp.fit_config)
print("\nCompile config:", hp.compile_config)
print("\nScheduler config:", hp.scheduler_config)
print("\nCross-validation config:", hp.cross_validation_config)

### 3.1 Loading from Files

`HyperParameter` can load from `.json` or `.py` files.

In [ ]:
import tempfile, os

# Save as JSON and reload
with tempfile.TemporaryDirectory() as tmpdir:
    json_path = os.path.join(tmpdir, "hyper.json")
    hp.save(json_path)
    
    hp_loaded = HyperParameter(json_path)
    print("Loaded from JSON:")
    print("  Model config:", hp_loaded.model_config)

In [ ]:
# Save as Python file and reload
with tempfile.TemporaryDirectory() as tmpdir:
    py_path = os.path.join(tmpdir, "hyper.py")
    with open(py_path, 'w') as f:
        f.write("hyper = " + repr(hyper) + "\n")
    
    hp_from_py = HyperParameter(py_path)
    print("Loaded from .py:")
    print("  Fit config:", hp_from_py.fit_config)

## 4. Full Training Workflow

This section demonstrates the complete config-driven training pipeline:
1. Load hyperparameters
2. Create model from config
3. Create optimizer from config
4. Create scheduler from config
5. Run `fit()` from `kgcnn_torch.training.trainer`

In [ ]:
import importlib
from kgcnn_torch.models.gcn import GCNModel
from kgcnn_torch.training.trainer import fit
from kgcnn_torch.training.scheduler import get_scheduler
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split


In [ ]:
# Step 1: Load real dataset from config
dataset_cfg = hp.dataset_config
dataset_class_name = dataset_cfg["class_name"]
dataset_module_name = dataset_cfg.get(
    "module_name", f"kgcnn_torch.data.datasets.{dataset_class_name}"
)
dataset_kwargs = dict(dataset_cfg.get("config", {}))
target_index = int(dataset_cfg.get("target_index", 0))

module = importlib.import_module(dataset_module_name)
dataset_class = getattr(module, dataset_class_name)
dataset = dataset_class(**dataset_kwargs)
print(f"Loaded {dataset_class_name} with {len(dataset)} graphs")

pyg_list = [dataset[i] for i in range(len(dataset))]
if len(pyg_list) == 0:
    raise RuntimeError("Dataset is empty.")

# Keep a single regression target for this GCN demo.
for data in pyg_list:
    if not hasattr(data, "y") or data.y is None:
        raise ValueError("Dataset samples must contain `y` labels for training.")
    y = data.y.detach().clone().reshape(-1).float()
    idx = min(target_index, y.numel() - 1)
    data.y = y[idx:idx + 1]

print(f"Prepared {len(pyg_list)} PyG graphs (target_index={target_index})")


In [ ]:
# Step 2: Convert to PyG and split
pyg_list = dataset.to_pyg_list()

indices = np.arange(len(pyg_list))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_data = [pyg_list[i] for i in train_idx]
val_data = [pyg_list[i] for i in val_idx]

# Use config for batch_size
batch_size = hp.fit_config.get("batch_size", 32)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size)

In [ ]:
# Step 3: Create model from config
model_config = hp.model_config
model = GCNModel(**model_config)
print(model)

In [ ]:
# Step 4: Create optimizer from config
compile_config = hp.compile_config
opt_config = compile_config.get("optimizer", {})
opt_params = opt_config.get("config", {"lr": 1e-3})

optimizer = torch.optim.Adam(model.parameters(), **opt_params)
print("Optimizer:", optimizer)

In [ ]:
# Step 5: Create scheduler from config
sched_config = hp.scheduler_config
scheduler = get_scheduler(
    name=sched_config["name"],
    optimizer=optimizer,
    **sched_config.get("config", {})
)
print("Scheduler:", scheduler)

In [ ]:
# Step 6: Run fit()
loss_fn = nn.MSELoss()
epochs = hp.fit_config.get("epochs", 100)

# Limit epochs for this demo
demo_epochs = min(epochs, 30)

history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    scheduler=scheduler,
    epochs=demo_epochs,
    verbose=1,
)

print("\nFinal train loss:", history["train_loss"][-1])
print("Final val loss:", history["val_loss"][-1])

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Train Loss")
if history["val_loss"]:
    axes[0].plot(history["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].set_title("Training Loss")

axes[1].plot(history["lr"], label="Learning Rate")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("LR")
axes[1].legend()
axes[1].set_title("Learning Rate Schedule")

plt.tight_layout()
plt.show()

## 5. Using Callbacks with fit()

The `fit()` function supports callbacks for checkpointing, early stopping, and LR logging.

In [ ]:
from kgcnn_torch.training.callbacks import (
    ModelCheckpointCallback,
    EarlyStoppingCallback,
    LearningRateLoggingCallback
)

# Reset model and optimizer for clean run
model2 = GCNModel(**hp.model_config)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)
scheduler2 = get_scheduler("warmup_exponential", optimizer2, warmup_epochs=5, decay_rate=0.96, decay_epochs=10)

callbacks = [
    EarlyStoppingCallback(patience=10, monitor="val_loss", verbose=1),
    LearningRateLoggingCallback(verbose=0),
]

history2 = fit(
    model=model2,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer2,
    loss_fn=nn.MSELoss(),
    scheduler=scheduler2,
    epochs=50,
    verbose=1,
    callbacks=callbacks,
)

print(f"Training completed after {len(history2['train_loss'])} epochs")

## Summary

| Component | Keras KGCNN | kgcnn_torch |
|---|---|---|
| Config class | `HyperParameter` | `HyperParameter` (same interface) |
| Config formats | .json, .py | .json, .py |
| Model creation | `make_model(**config)` | `ModelClass(**config)` (native nn.Module) |
| Optimizer | `ks.optimizers.Adam(...)` | `torch.optim.Adam(...)` |
| Scheduler | Keras LR callbacks | `get_scheduler()` returning PyTorch schedulers |
| Training loop | `model.fit()` | `kgcnn_torch.training.trainer.fit()` |
| Callbacks | Keras callbacks | `TrainingCallback` subclasses |

The config structure is deliberately kept similar to Keras KGCNN so that existing hyperparameter files can be adapted with minimal changes.